# Train YOLO11n: hộp, chân bàn/ghế và dép

**Dữ liệu tổng hợp từ camera trước gắn trên xe trong Webots**, 1.200 cảnh độc lập:
840 train / 180 validation / 180 test. Lớp: `box`, `furniture_leg`, `slipper`.
Mỗi chân bàn/ghế có nhãn riêng; mặt bàn/ghế là vật che khuất, không phải nhãn chân.

1. Chọn **Runtime → Change runtime type → GPU** (T4 nếu có).
2. Chạy các ô theo thứ tự. Bộ dữ liệu sẽ được tải tự động từ repo.
3. Sau train, chạy ô test một lần, rồi tải ZIP kết quả gồm `best.pt`, `last.pt`, metrics và biểu đồ.

Chưa có model mới được train trước trong notebook này. GPU Colab phụ thuộc tài nguyên/quota tài khoản.
[Colab FAQ](https://research.google.com/colaboratory/faq.html) ·
[Ultralytics train](https://docs.ultralytics.com/modes/train/)

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.202'])
import torch, ultralytics
assert torch.cuda.is_available(), 'Chưa có GPU: chọn Runtime > Change runtime type > GPU rồi chạy lại.'
print('GPU:', torch.cuda.get_device_name(0))
print('Torch:', torch.__version__, '| CUDA:', torch.version.cuda, '| Ultralytics:', ultralytics.__version__)

## Lấy bộ dữ liệu đã chuẩn bị

Mặc định tải file **`robot_objects_v1_dataset.zip`** từ repo project rồi kiểm tra SHA-256 trước khi giải nén.
Không cần upload thủ công. Nếu mạng GitHub gặp lỗi, đổi `USE_LOCAL_UPLOAD=True` và chọn ZIP trên máy.
Chưa cần gán nhãn lại.

In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, zipfile, json, urllib.request
USE_LOCAL_UPLOAD = False
if USE_LOCAL_UPLOAD:
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith('.zip')]
    assert len(zip_names) == 1, 'Chọn đúng một ZIP dữ liệu.'
    archive_path = Path(zip_names[0])
else:
    archive_path = Path('/content/robot_objects_v1_dataset.zip')
    urllib.request.urlretrieve('https://raw.githubusercontent.com/hung230206/indoor-autonomous-robot/main/artifacts/training/robot_objects_v1_dataset.zip', archive_path)
DATASET_ZIP_SHA256 = 'ed4877d4ca019968d0f09a77e531aef36c72bf91ccd8ffbcd1717004592418f5'
assert hashlib.sha256(archive_path.read_bytes()).hexdigest() == DATASET_ZIP_SHA256, 'Sai ZIP hoặc file bị hỏng.'
extract_root = Path('/content/robot_objects_v1')
assert not extract_root.exists(), 'Thư mục dữ liệu đã tồn tại. Dùng runtime mới để tránh trộn dữ liệu.'
with zipfile.ZipFile(archive_path) as archive:
    for name in archive.namelist():
        part = PurePosixPath(name)
        assert not part.is_absolute() and '..' not in part.parts, 'Đường dẫn ZIP không hợp lệ.'
    archive.extractall(extract_root)
data_root = extract_root / 'dataset'
print('Dataset:', data_root)

In [ ]:
sys.path.insert(0, str(data_root))
from validate_object_dataset import validate_dataset
from train_objects import train_model, evaluate_model, export_results
integrity = validate_dataset(data_root, minimum_class_images=30)
print(json.dumps(integrity, indent=2))
# Rewrite only the portable YAML root for this Colab VM.
from ultralytics.utils import YAML
dataset_config = YAML.load(data_root / 'dataset.yaml')
dataset_config['path'] = str(data_root)
YAML.save(data_root / 'dataset.yaml', dataset_config)

## Huấn luyện trên GPU

Mặc định 60 epoch, ảnh 640, batch 16, YOLO11n COCO, seed 42. Ảnh 640 giữ chi tiết chân mảnh tốt hơn
ảnh 320 của bản cũ nhưng cần đo lại tốc độ khi tích hợp. Chọn checkpoint theo validation, không theo test.

`SAVE_TO_DRIVE=False` lưu trong máy ảo; nhớ tải kết quả trước khi mất phiên. Có thể bật `True` để giữ
checkpoint trên Drive: Colab sẽ yêu cầu **bạn** cấp quyền. Thay batch thành 8 nếu GPU báo hết bộ nhớ.

In [ ]:
from datetime import datetime, timezone
SAVE_TO_DRIVE = False
EPOCHS, IMAGE_SIZE, BATCH = 60, 640, 16
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    project = Path('/content/drive/MyDrive/robot_yolo_training')
else:
    project = Path('/content/robot_yolo_training')
run_name = 'objects_v1_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
run = train_model(data_root, project, run_name, epochs=EPOCHS,
                  imgsz=IMAGE_SIZE, batch=BATCH, device='0')
provenance_path = run / 'training-provenance.json'
provenance = json.loads(provenance_path.read_text())
provenance['dataset_zip_sha256'] = DATASET_ZIP_SHA256
provenance_path.write_text(json.dumps(provenance, indent=2))
print('Checkpoint được chọn từ validation:', run / 'weights/best.pt')

In [ ]:
from IPython.display import Image, display
if (run / 'results.png').exists():
    display(Image(filename=str(run / 'results.png')))

## Đánh giá test sau khi đã chọn checkpoint

Ô này đánh giá 180 cảnh test và ghi precision, recall, mAP50, mAP50–95 cho từng lớp.
Chỉ dùng test để báo kết quả cuối; nếu điều chỉnh theo lỗi test, lần sau cần một tập giữ lại mới.
Chỉ số này **không so ngang** mAP của bộ box cũ, không chứng minh xe tránh được mọi vật và không phải ảnh thật.

In [ ]:
report = evaluate_model(data_root, run, device='0', imgsz=IMAGE_SIZE, batch=BATCH, split='test')
print(json.dumps(report, indent=2))
if (run / 'test_evaluation/confusion_matrix.png').exists():
    display(Image(filename=str(run / 'test_evaluation/confusion_matrix.png')))

## Lưu kết quả về máy

Giữ cả ZIP kết quả, không chỉ `best.pt`. Sau đó đưa ZIP về project để kiểm tra đúng thứ tự lớp,
tích hợp detector đa lớp và chạy lại các ca bàn/ghế/dép. Notebook không tự thay model demo.
Phép chiếu mép dưới bbox vẫn có thể sai khi chân bị che/cắt; train detector chưa tự sửa được giới hạn đó.

In [ ]:
result_zip = export_results(run, Path('/content') / (run.name + '_results.zip'))
print('Kết quả:', result_zip)
files.download(str(result_zip))